In [ ]:
# ============================================================
# SignLingo Round 2 -- Phase 2: run_ablation
# ============================================================
# The long pole. Reads dataset.npz from prepare_features and answers, in order:
#
#   1  Is the LOSO harness sound, or does the drop come from the new code?
#      (Next-Steps Step 1, a GATE -- the grid does not run if this fails.)
#   2  What does a sequence-level split score on the same data, for reference?
#   3  What does each feature block contribute to signer-independent accuracy?
#      Four arms, each a prefix of the compact vector, four LOSO folds each.
#
# Every arm sees identical folds, identical augmentation, identical epochs and
# identical hyperparameters. Only the feature width changes (Next-Steps 5.4).
#
# Results append to phase2_results.json after every fold, and a re-run skips
# whatever is already in there. This is meant to be launched unattended on
# another machine, where a crash at run 15 of 21 must not cost the whole night.
import os
import gc
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, confusion_matrix)

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
DATA_PATH    = "dataset.npz"
RESULTS_PATH = "phase2_results.json"
RANDOM_STATE = 42

# Frozen, NOT re-tuned. These came from the 178-trial Hyperband search run under a
# sequence-level split, so they were optimised for a protocol that permits signer
# memorisation. Re-tuning them on LOSO folds would be selecting on test, the exact
# error this whole round exists to fix, so they stay put and the paper carries the
# limitation sentence from Next-Steps 5.5 instead.
GRU1, GRU2, DROPOUT, L2_RATE, USE_REC_L2, LR = 32, 256, 0.3, 1e-4, False, 1e-3
EPOCHS       = 150
BATCH_SIZE   = 32
PATIENCE     = 15
AUG_FACTOR   = 3            # train-only, applied inside each fold
MIRROR_PROB  = 0.5          # per-sample horizontal flip, ON for every arm
VAL_FRACTION = 0.12

# The gate. Next-Steps Step 1 prefers 3 seeds; 1 is the honest minimum and costs
# 4 runs instead of 12. Raise it if the single draw lands ambiguously.
N_PERMUTED_SEEDS = 1
GATE_MIN_ACC     = 0.90

# Integration-test switch. Shrinks everything so all 21 runs finish in minutes on
# real data and real splits, which is how the Round 1 axis bugs were caught before
# they could cost an overnight slot. Set False for the real run.
TOY_MODE = False
if TOY_MODE:
    # AUG_FACTOR stays at 2, not 1: at 1 the augmentation loop body never runs and
    # the integration test would skip the augment-and-mirror path entirely, which
    # is one of the paths most worth exercising.
    GRU1, GRU2, EPOCHS, PATIENCE, AUG_FACTOR = 4, 8, 1, 1, 2
    N_PERMUTED_SEEDS, GATE_MIN_ACC = 1, 0.0
    RESULTS_PATH = "phase2_results_toy.json"

In [ ]:
# ============================================================
# 1 -- Load the prepared dataset
# ============================================================
d = np.load(DATA_PATH, allow_pickle=True)
X            = d["X"].astype(np.float32)
y_int        = d["y"].astype(int)
groups       = d["groups"]
class_names  = [str(c) for c in d["class_names"]]
arm_names    = [str(a) for a in d["arm_names"]]
arm_cuts     = [int(c) for c in d["arm_cuts"]]
nms_mask     = d["nms_mask"]
mirror_perm  = d["mirror_perm"]
mirror_sign  = d["mirror_sign"].astype(np.float32)

num_classes = len(class_names)
y_oh = to_categorical(y_int, num_classes).astype(np.float32)
signers = sorted(np.unique(groups))
ARMS = list(zip(arm_names, arm_cuts))

print("Loaded %s: %d sequences, %d classes, %d signers"
      % (DATA_PATH, len(X), num_classes, len(signers)))
print("Arms: " + ", ".join("%s(%d)" % (n, c) for n, c in ARMS))
print("NMS-focused classes: %d of %d" % (nms_mask.sum(), num_classes))
assert len(signers) >= 2, "LOSO needs at least 2 signers in dataset.npz"
assert arm_cuts[-1] == X.shape[-1], "arm cuts disagree with the feature width"


def mirror(a):
    """Signed permutation recovered in prepare_features, applied verbatim. The
    mirror is defined there once; nothing about the block layout is repeated here."""
    return a[..., mirror_perm] * mirror_sign


assert np.allclose(mirror(mirror(X[:8])), X[:8], atol=1e-5), \
    "mirroring twice is not the identity; dataset.npz is inconsistent"
print("Mirror round-trips on real sequences.")

In [ ]:
# ============================================================
# 2 -- Augmentation, block-aware
# ============================================================
# The Round 1 augmenter reshaped the whole vector to (frames, -1, 3) and applied a
# 3D rotation and a scale factor to all of it. That is wrong for the compact
# vector: only block A is coordinates. Blocks B, C and D are unit vectors, angles
# and dimensionless ratios, and 146 is not even divisible by 3.
#
# What is legitimate per block:
#   in-plane rotation  A and B rotate, tilt and roll shift by the angle. Camera
#                      tilt is real and shoulder normalisation does not remove it,
#                      since it translates and scales but never rotates.
#   scale jitter       A only. B, C and D are scale-free by construction, so
#                      scaling them manufactures data the extractor cannot emit.
#   noise              everything, then block B is re-normalised to unit length.
#   time resample      everything.
A_END, B_END = 126, 138          # block A coords, block B limb unit vectors
I_TILT, I_ROLL = 138, 139


def _renorm_limbs(a):
    v = a[:, A_END:B_END].reshape(len(a), 4, 3)
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    a[:, A_END:B_END] = (v / np.maximum(n, 1e-6)).reshape(len(a), B_END - A_END)
    return a


def augment_sequence(seq, rng):
    n_frames, n_feat = seq.shape
    out = seq.astype(np.float32).copy()

    # temporal resample first, so later per-frame edits are not re-interpolated
    new_len = n_frames * rng.uniform(0.9, 1.1)
    src = np.linspace(0, n_frames - 1, n_frames)
    dst = np.clip(np.linspace(0, new_len - 1, n_frames), 0, n_frames - 1)
    out = np.stack([np.interp(dst, src, out[:, f]) for f in range(n_feat)], axis=1)

    th = np.radians(rng.uniform(-5, 5))
    c, s = np.cos(th), np.sin(th)
    R2 = np.array([[c, -s], [s, c]], dtype=np.float32)
    for lo, hi in ((0, A_END), (A_END, B_END)):
        v = out[:, lo:hi].reshape(n_frames, -1, 3)
        v[..., :2] = v[..., :2] @ R2.T
        out[:, lo:hi] = v.reshape(n_frames, hi - lo)
    out[:, I_TILT] += th
    out[:, I_ROLL] += th

    out[:, 0:A_END] *= np.float32(rng.uniform(0.95, 1.05))
    out += rng.normal(0, 0.005, out.shape).astype(np.float32)
    return _renorm_limbs(out).astype(np.float32)


# A self-check, because a silently wrong augmenter poisons every arm equally and
# so never shows up as a suspicious difference between them.
_rng = np.random.default_rng(0)
_a = augment_sequence(X[0], _rng)
assert _a.shape == X[0].shape
_n = np.linalg.norm(_a[:, A_END:B_END].reshape(-1, 4, 3), axis=-1)
assert np.allclose(_n, 1.0, atol=1e-4), "augmentation broke the block B unit norm"
assert not np.allclose(_a, X[0]), "augmentation changed nothing"
# rotation must move blocks A/B/tilt/roll but leave the scale-free ratios alone
_z = augment_sequence(X[0], np.random.default_rng(0))
assert np.allclose(_a, _z), "augmentation is not reproducible from its seed"
print("Augmentation self-check passed (unit norm preserved, seed reproducible).")


def build_training_pool(Xtr, ytr, seed):
    """x AUG_FACTOR, train split only, with mirroring folded in. Mirror is ON for
    every arm: a different augmentation per arm would break the comparison."""
    rng = np.random.default_rng(seed)
    pool_x, pool_y = [Xtr], [ytr]
    for _ in range(max(0, AUG_FACTOR - 1)):
        aug = np.stack([augment_sequence(s, rng) for s in Xtr])
        flip = rng.random(len(aug)) < MIRROR_PROB
        aug[flip] = mirror(aug[flip])
        pool_x.append(aug)
        pool_y.append(ytr)
    Xp, yp = np.concatenate(pool_x), np.concatenate(pool_y)
    idx = rng.permutation(len(Xp))
    return Xp[idx], yp[idx]

In [ ]:
# ============================================================
# 3 -- Model and one train/evaluate cycle
# ============================================================
def build_model(input_dim):
    kreg = regularizers.l2(L2_RATE)
    rreg = regularizers.l2(L2_RATE) if USE_REC_L2 else None
    m = Sequential([
        Input(shape=(X.shape[1], input_dim)),
        GRU(GRU1, return_sequences=True, name="gru_1",
            kernel_regularizer=kreg, recurrent_regularizer=rreg),
        Dropout(DROPOUT, name="dropout_1"),
        GRU(GRU2, return_sequences=False, name="gru_2",
            kernel_regularizer=kreg, recurrent_regularizer=rreg),
        Dropout(DROPOUT, name="dropout_2"),
        Dense(num_classes, activation="softmax", name="output",
              kernel_regularizer=regularizers.l2(L2_RATE)),
    ], name="SignLingo_GRU_R2")
    m.compile(optimizer=Adam(learning_rate=LR),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m


def train_and_eval(Xtr, ytr, Xva, yva, Xte, yte, tag=""):
    tf.keras.backend.clear_session()
    gc.collect()
    t0 = time.time()
    model = build_model(Xtr.shape[-1])
    hist = model.fit(
        Xtr, ytr, validation_data=(Xva, yva),
        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=PATIENCE,
                          restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6,
                              min_lr=1e-6, verbose=0),
        ])
    prob = model.predict(Xte, verbose=0)
    pred, true = prob.argmax(1), yte.argmax(1)
    row = {
        "tag": tag,
        "accuracy":  float(accuracy_score(true, pred)),
        "precision": float(precision_score(true, pred, average="weighted", zero_division=0)),
        "recall":    float(recall_score(true, pred, average="weighted", zero_division=0)),
        "f1":        float(f1_score(true, pred, average="weighted", zero_division=0)),
        "loss":      float(log_loss(true, prob, labels=list(range(num_classes)))),
        "epochs_run": int(len(hist.history["loss"])),
        "params": int(model.count_params()),
        "minutes": round((time.time() - t0) / 60, 2),
        "pred": pred.tolist(),
        "true": true.tolist(),
    }
    # The 20/20 split from Next-Steps 5.3, free: just a slice of the predictions.
    # If block C really contributes, the gain concentrates in the NMS-focused half
    # rather than spreading evenly over all 40 classes.
    for name, sel in (("nms", nms_mask[true]), ("manual", ~nms_mask[true])):
        row["acc_" + name] = (float(accuracy_score(true[sel], pred[sel]))
                              if sel.any() else None)
    del model
    return row


# Results accumulate on disk. Anything already recorded is skipped on a re-run.
results = {"config": {}, "permuted_control": {}, "stratified_reference": None, "arms": {}}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as fh:
        results.update(json.load(fh))
    print("Resuming from %s" % RESULTS_PATH)
new_config = {
    "gru1": GRU1, "gru2": GRU2, "dropout": DROPOUT, "l2": L2_RATE,
    "use_recurrent_l2": USE_REC_L2, "lr": LR, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "patience": PATIENCE, "aug_factor": AUG_FACTOR,
    "mirror_prob": MIRROR_PROB, "arms": dict(ARMS), "toy_mode": TOY_MODE,
    "hyperparameters_frozen_from": "178-trial Hyperband under a sequence-level split",
}
# Resuming with different settings would mix folds trained under different configs
# into one results file, and nothing downstream could tell them apart. Refuse it.
_old = results.get("config") or {}
_diff = [k for k, v in new_config.items() if k in _old and _old[k] != v]
assert not _diff, (
    "%s was written with different settings (%s differ). Folds trained under two "
    "configs cannot be compared. Delete the file to start clean, or restore the "
    "old settings to continue it." % (RESULTS_PATH, ", ".join(_diff)))
results["config"] = new_config


def save():
    with open(RESULTS_PATH, "w") as fh:
        json.dump(results, fh, indent=2)


def fold_split(tr_idx, te_idx, cut, seed):
    """One fold, from index sets to model-ready tensors. Augmentation happens
    after the validation split, so no augmented copy of a validation sequence can
    leak into training."""
    Xtr_raw, ytr_raw = X[tr_idx], y_oh[tr_idx]
    Xtr_raw, Xva, ytr_raw, yva = train_test_split(
        Xtr_raw, ytr_raw, test_size=VAL_FRACTION,
        stratify=y_int[tr_idx], random_state=seed)
    Xtr, ytr = build_training_pool(Xtr_raw, ytr_raw, seed)
    return (Xtr[..., :cut], ytr, Xva[..., :cut], yva,
            X[te_idx][..., :cut], y_oh[te_idx])

In [ ]:
# ============================================================
# 4 -- GATE: permuted-group control
# ============================================================
# Moving from 5-fold to LOSO changed two things at once: the folds became
# signer-disjoint, and the harness became new code. A drop could come from either.
# This runs the identical harness with the signer labels randomly permuted while
# holding every fold size fixed, so grouping is the only variable that changes.
#
#   ~99%  harness is sound, the LOSO drop is a real signer effect  -> continue
#   ~67%  something leaks or is broken in the harness              -> stop
#
# It asserts rather than prints, so an unattended run cannot spend the night
# building a 16-run grid on top of a broken foundation.
logo = LeaveOneGroupOut()
full_cut = arm_cuts[-1]

for seed_i in range(N_PERMUTED_SEEDS):
    key = "seed_%d" % seed_i
    if key in results["permuted_control"]:
        print("Permuted control %s already done, skipping." % key)
        continue
    rng = np.random.default_rng(RANDOM_STATE + seed_i)
    # Permute the labels, not the sizes: each pseudo-signer keeps exactly the
    # sequence count its real counterpart had, so fold sizes are untouched.
    fake = np.empty_like(groups)
    order = rng.permutation(len(groups))
    at = 0
    for s in signers:
        n = int((groups == s).sum())
        fake[order[at:at + n]] = s
        at += n
    assert all((fake == s).sum() == (groups == s).sum() for s in signers)

    rows = []
    for k, (tr, te) in enumerate(logo.split(X, y_int, fake)):
        r = train_and_eval(*fold_split(tr, te, full_cut, RANDOM_STATE),
                           tag="permuted_%d_fold%d" % (seed_i, k))
        rows.append(r)
        print("  permuted seed %d fold %d: %.2f%%  (%d epochs, %.1f min)"
              % (seed_i, k, r["accuracy"] * 100, r["epochs_run"], r["minutes"]))
    results["permuted_control"][key] = {
        "folds": rows,
        "mean_accuracy": float(np.mean([r["accuracy"] for r in rows])),
    }
    save()

gate = float(np.mean([v["mean_accuracy"] for v in results["permuted_control"].values()]))
print("\nPermuted-group control: %.2f%% (threshold %.0f%%)"
      % (gate * 100, GATE_MIN_ACC * 100))
assert gate >= GATE_MIN_ACC, (
    "GATE FAILED: randomly grouped folds scored %.2f%%, below %.2f%%. The harness "
    "itself is losing accuracy, so any LOSO drop cannot be attributed to signer "
    "identity. Debug the split, the augmentation, or the training length before "
    "running the grid." % (gate * 100, GATE_MIN_ACC * 100))
if TOY_MODE:
    print("TOY MODE: the gate threshold is 0, so this proves only that the control "
          "RUNS.\nIt validates nothing about the harness. The real run must clear "
          "%.0f%%." % (GATE_MIN_ACC * 100 or 90))
else:
    print("GATE PASSED. Fold structure is sound, so the LOSO drop is a signer effect.")

In [ ]:
# ============================================================
# 5 -- Stratified reference
# ============================================================
# One sequence-level run on the same cleaned data, for a like-for-like comparison
# against the paper's 99.32%. That figure was measured with duplicates still in
# and with the old 447-dim features, so neither number transfers directly.
if results["stratified_reference"] is None:
    tr, tmp = train_test_split(np.arange(len(X)), test_size=0.2,
                               stratify=y_int, random_state=RANDOM_STATE)
    va, te = train_test_split(tmp, test_size=0.5,
                              stratify=y_int[tmp], random_state=RANDOM_STATE)
    Xtr, ytr = build_training_pool(X[tr], y_oh[tr], RANDOM_STATE)
    results["stratified_reference"] = train_and_eval(
        Xtr[..., :full_cut], ytr, X[va][..., :full_cut], y_oh[va],
        X[te][..., :full_cut], y_oh[te], tag="stratified_reference")
    save()
print("Stratified reference (full arm): %.2f%%"
      % (results["stratified_reference"]["accuracy"] * 100))

In [ ]:
# ============================================================
# 6 -- Main grid: 4 arms x 4 LOSO folds
# ============================================================
# Every arm sees the same folds, the same augmentation and the same epoch budget.
# The only thing that varies is how many columns of the vector the model can see,
# so each step of the ladder isolates one block.
fold_ids = [(int(te[0]), str(groups[te][0])) for _, te in logo.split(X, y_int, groups)]
print("\nFolds: " + ", ".join(s for _, s in fold_ids))

for arm_name, cut in ARMS:
    results["arms"].setdefault(arm_name, {"cut": cut, "folds": {}})
    print("\n" + "=" * 60)
    print("ARM %s  (%d dims)" % (arm_name, cut))
    print("=" * 60)
    for tr, te in logo.split(X, y_int, groups):
        held = str(groups[te][0])
        if held in results["arms"][arm_name]["folds"]:
            print("  %-22s already done, skipping." % held)
            continue
        r = train_and_eval(*fold_split(tr, te, cut, RANDOM_STATE),
                           tag="%s_%s" % (arm_name, held))
        r["held_out_signer"] = held
        results["arms"][arm_name]["folds"][held] = r
        save()
        print("  %-22s %.2f%%   nms %s  manual %s   (%d ep, %.1f min)"
              % (held, r["accuracy"] * 100,
                 "n/a" if r["acc_nms"] is None else "%.1f%%" % (r["acc_nms"] * 100),
                 "n/a" if r["acc_manual"] is None else "%.1f%%" % (r["acc_manual"] * 100),
                 r["epochs_run"], r["minutes"]))

for arm_name, _ in ARMS:
    a = results["arms"][arm_name]
    accs = [a["folds"][s]["accuracy"] for s in sorted(a["folds"])]
    a["mean_accuracy"] = float(np.mean(accs))
    a["std_accuracy"] = float(np.std(accs))
save()

In [ ]:
# ============================================================
# 7 -- Summary: direction and magnitude per fold, not a p-value
# ============================================================
# With n=4 folds a paired t-test has almost no power and its p-value is fragile.
# Consistency of direction plus per-fold magnitude is the stronger evidence, and
# it is what a reviewer can actually check (Next-Steps 5.6).
order = sorted(results["arms"][ARMS[0][0]]["folds"])
print("\n" + "=" * 74)
print("PHASE 2 SUMMARY")
print("=" * 74)
print("Permuted-group control : %.2f%%   (gate, must be high)" % (gate * 100))
print("Stratified reference   : %.2f%%   (sequence-level split, same data)"
      % (results["stratified_reference"]["accuracy"] * 100))
print()
hdr = "%-16s %5s" % ("arm", "dim") + "".join("%14s" % s.replace("Signer_", "")
                                             for s in order) + "%12s%9s" % ("mean", "std")
print(hdr)
for arm_name, cut in ARMS:
    a = results["arms"][arm_name]
    print("%-16s %5d" % (arm_name, cut)
          + "".join("%13.2f%%" % (a["folds"][s]["accuracy"] * 100) for s in order)
          + "%11.2f%%%8.2f%%" % (a["mean_accuracy"] * 100, a["std_accuracy"] * 100))

print("\nPer-block contribution (each row adds exactly one block):")
for i in range(1, len(ARMS)):
    prev, cur = ARMS[i - 1][0], ARMS[i][0]
    deltas = [(results["arms"][cur]["folds"][s]["accuracy"]
               - results["arms"][prev]["folds"][s]["accuracy"]) * 100 for s in order]
    same = "all %d folds" % len(deltas) if all(np.sign(d) == np.sign(deltas[0])
                                               for d in deltas) else "MIXED direction"
    print("  %-16s -> %-16s %+6.2f pp   per fold %s   %s"
          % (prev, cur, np.mean(deltas),
             "[" + ", ".join("%+.1f" % v for v in deltas) + "]", same))

print("\nNMS-focused vs standard-manual (20/20, pre-registered):")
print("%-16s %12s %12s %10s" % ("arm", "NMS", "manual", "gap"))
for arm_name, _ in ARMS:
    a = results["arms"][arm_name]
    n = np.mean([a["folds"][s]["acc_nms"] for s in order])
    m = np.mean([a["folds"][s]["acc_manual"] for s in order])
    print("%-16s %11.2f%% %11.2f%% %9.2f pp" % (arm_name, n * 100, m * 100,
                                                (n - m) * 100))
print("\nIf a block genuinely carries non-manual information, its gain should sit")
print("in the NMS-focused half rather than spread evenly across all 40 classes.")
save()

In [ ]:
# ============================================================
# 8 -- Charts
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
w = 0.8 / len(ARMS)
xp = np.arange(len(order))
for k, (arm_name, _) in enumerate(ARMS):
    a = results["arms"][arm_name]
    ax[0].bar(xp + (k - (len(ARMS) - 1) / 2) * w,
              [a["folds"][s]["accuracy"] * 100 for s in order], w, label=arm_name)
ax[0].set_xticks(xp)
ax[0].set_xticklabels([s.replace("Signer_", "") for s in order], fontsize=8)
ax[0].set_ylabel("LOSO accuracy (%)")
ax[0].set_title("Accuracy by held-out signer and arm", fontweight="bold")
ax[0].legend(fontsize=8)

means = [results["arms"][n]["mean_accuracy"] * 100 for n, _ in ARMS]
stds = [results["arms"][n]["std_accuracy"] * 100 for n, _ in ARMS]
ax[1].bar([n for n, _ in ARMS], means, yerr=stds, capsize=5, color="#4C72B0")
ax[1].axhline(gate * 100, color="#C44E52", ls="--", lw=1,
              label="permuted control %.1f%%" % (gate * 100))
ax[1].set_ylabel("mean LOSO accuracy (%)")
ax[1].set_title("Mean across folds (error bars = std)", fontweight="bold")
ax[1].tick_params(axis="x", rotation=20, labelsize=8)
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("phase2_arms.png", dpi=120)
plt.show()

best = max(ARMS, key=lambda a: results["arms"][a[0]]["mean_accuracy"])
cm_rows = []
for s in order:
    f = results["arms"][best[0]]["folds"][s]
    cm_rows.append(confusion_matrix(f["true"], f["pred"],
                                    labels=list(range(num_classes))))
cm = np.sum(cm_rows, axis=0)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, ax=ax, cmap="mako", cbar=True, square=True,
            xticklabels=class_names, yticklabels=class_names)
ax.set_title("Confusion matrix, %s, all LOSO folds pooled" % best[0], fontweight="bold")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.savefig("phase2_confusion.png", dpi=120)
plt.show()

results["best_arm"] = {"name": best[0], "cut": best[1],
                       "mean_accuracy": results["arms"][best[0]]["mean_accuracy"]}
save()
print("\nBest arm: %s (%d dims) at %.2f%% mean LOSO."
      % (best[0], best[1], results["arms"][best[0]]["mean_accuracy"] * 100))
print("train_final reads this straight from %s, so no constant needs editing."
      % RESULTS_PATH)
print("Wrote %s, phase2_arms.png, phase2_confusion.png" % RESULTS_PATH)